In [ ]:
# imports
import spacy
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from collections import defaultdict
import requests

# HuggingFace API
from huggingface_hub import InferenceClient
from openai import OpenAI
from urllib import response

# load environment variables
from dotenv import load_dotenv
import os

In [82]:
# old functions

# # Entity and relation extraction functions
# def get_quantity(noun):
#     for child in noun.children:
#         if child.pos_ in ("DET", "NUM"):
#             return normalize_quantity(child, noun)
#     return "m"  # default

# # Quantity normalization function
# def normalize_quantity(token, noun):
#     quantity_map = {
#         "ein": "1",
#         "eine": "1",
#         "jede": "1",
#         "jeder": "1",
#         "jedes": "1",
#         "mehrere": "n",
#         "viele": "n",
#         "alle": "n",
#         "einige": "n",
#         "kein": "0",
#         "keine": "0",
#     }

#     if token.pos_ == "NUM":
#         return token.text
#     elif noun.morph.get("Number") == ["Plur"]:
#         return "n"
#     elif token.pos_ == "DET":
#         text_lower = token.text.lower()
#         return quantity_map.get(text_lower, "m")
#     return "m"


# # Cardinality extraction function
# def get_cardinality(noun, modal=None):
#     """
#     Gibt (min, max) zurück basierend auf:
#     - Determinator
#     - Morphologie  
#     - Modalverb
#     - Numerale
#     """
#     det = None
#     num = None
    
#     for child in noun.children:
#         if child.pos_ == "DET":
#             det = child.text.lower()
#         elif child.pos_ == "NUM":
#             num = child.text

#     max_card = get_max(det, num, noun)
#     # min_card = get_min(det, num, noun, modal)
    
#     return (max_card)
#     return (dict(min=min_card), dict(max=max_card))

# # cardinality max function
# def get_max(det, num, noun):
#     if num:
#         return num  # z.B. "3"
#     if det in ("mehrere", "viele", "alle", "einige"):
#         return "n"
#     if noun.morph.get("Number") == ["Plur"]:
#         return "n"
#     return "1"  # Singular ohne Spezifikation

# # cardinality min function
# def get_min(det, num, noun, modal):
#     # Optionalität durch Modal
#     if modal in ("können", "dürfen"):
#         return "0"
#     if det in ("kein", "keine"):
#         return "0"
#     if num:
#         return num
#     #if det in ("mehrere", "einige"):
#     #    return "n"
#     return "1"  # Default


# def get_modal(sent):
#     """
#     Extrahiert Modalverb aus dem sent falls vorhanden
#     """
#     for token in sent:
#         if token.pos_ == "AUX" and token.lemma_ in ("können", "müssen", "dürfen", "sollen", "wollen", "mögen"):
#             return token.lemma_
#     return None


In [83]:
# function declarations: filter

def is_relevant(sent):
    has_noun_subject = any(
        t.dep_ in ("sb", "nsubj") and t.pos_ == "NOUN" 
        for t in sent
    )
    
    has_kardinalitaet = any(
        t.text.lower() in ["jeder", "jede", "jedes", "mehrere", 
                           "verschiedene", "kein", "alle", "eine", "einen"]
        for t in sent
    )
    
    has_verb = any(t.pos_ in ("VERB", "AUX") for t in sent)
    
    noun_count = sum(1 for t in sent if t.pos_ == "NOUN")
    
    # relevant if: subject + verb + either cardinality or multiple nouns
    return has_noun_subject and has_verb and (has_kardinalitaet or noun_count >= 2)

# checks if sentence has a verb as root
def has_root_verb(sent):
    has_root_verb = None
    
    for token in sent:
        if token.pos_ in ("VERB", "AUX") and token.dep_ == "ROOT":
            has_root_verb = True
            break
    return has_root_verb

def phrase_replacer(sent):
    PHRASEME = {"in der Regel", "in der Tat", "in Ordnung", "auf jeden Fall"}

    from spacy.matcher import PhraseMatcher
    matcher = PhraseMatcher(nlp.vocab)
    matcher.add("PHRASEM", [nlp(p) for p in PHRASEME])


In [84]:
# function declarations rulebased extraction

# Entity management functions
def get_entity_id(noun, entities):
    global entity_id_counter
    key = noun.lemma_
    if key not in entities:
        entities[key] = {
            "id": f"e{entity_id_counter}",
            "type": noun.lemma_,
        }
        entity_id_counter += 1
    return entities[key]["id"]

# Verb extraction function
# def get_main_verb(sent):
#     if [token for token in sent if token.pos_ == "VERB"]:
#         return [token.lemma_ for token in sent if token.pos_ == "VERB"][0]
#     root = sent.root

#     # If ROOT is a modal or auxiliary verb, look for the main verb in its children
#     if root.pos_ in ["AUX", "VERB"]:
#         for child in root.children:
#             if child.dep_ in ["xcomp", "oc", "pd"] and child.pos_ == "VERB":
#                 return child.lemma_

#     return root.lemma_


def get_verb_root(token):
    """Findet das oberste Verb eines Verbalkomplexes"""
    # Aufsteigen bis ROOT oder rc
    head = token.head
    while head.dep_ not in ("ROOT", "rc"):
        head = head.head
    
    # Absteigen zum Hauptverb
    def find_main_verb(node):
        for child in node.children:
            if child.pos_ in ("VERB", "AUX"): # child.dep_ in ("oc", "pd") and 
                return find_main_verb(child)
        return node.i
    
    return find_main_verb(head)


def get_conjunctions(node, verb_groups, verb_idx, role):
    """Rekursiv alle konjunktiv verbundenen Nomen sammeln"""
    verb_groups[verb_idx][role].append(node)
    for child in node.children:
        if child.dep_ == "cj" and child.pos_ == "NOUN":
            get_conjunctions(child, verb_groups, verb_idx, role)
        if child.dep_ == "cd" and child.pos_ == "CCONJ":
            for conj_child in child.children:
                if conj_child.pos_ == "NOUN" and conj_child.dep_ in ("nk", "da", "cj"):
                    get_conjunctions(conj_child, verb_groups, verb_idx, role)


def get_subjects(token, verb_groups, verb_idx):
    if token.pos_ == "PRON":
        head = token.head
        while head.dep_ not in ("rc", "ROOT"):
            head = head.head
        if head.dep_ == "rc":
            subj = head.head
            get_conjunctions(subj, verb_groups, verb_idx, "subjects")
    else:
        get_conjunctions(token, verb_groups, verb_idx, "subjects")

def get_objects(token, verb_groups, verb_idx):
    get_conjunctions(token, verb_groups, verb_idx, "objects")


# Cardinality extraction function
def get_cardinality(token):
    for child in token.children:
        if child.lemma_ in ("ein", "eine"):
            return "1"
        if child.lemma_ in ("mehrere", "viele", "alle", "einige"):
            return "n"
        if child.dep_ == "det":
            # morphology check for determiner
            number = child.morph.get("Number")
            if number == ["Plur"]:
                return "n"
            if number == ["Sing"]:
                return "1"
    # Fallback: morphology check for noun itself
    number = token.morph.get("Number")
    if number == ["Plur"]:
        return "n"
    return "1_fallback"


# def extract_relations(sent):
#     relations = []
    
#     for token in sent:
#         # only consider verbs as starting point
#         if token.pos_ not in ("VERB", "AUX"):
#             continue
        
#         subjects = [c for c in token.children if c.dep_ == "sb" and c.pos_ == "NOUN"]
#         if not subjects:
#             continue
#         objects = [c for c in token.children if c.dep_ in ("oa", "op", "od") and c.pos_ == "NOUN"]

#         subjekt = subjects[0]
#         all_objects = []

#         for obj in objects:
#             all_objects.append(obj)
#             for child in obj.children:
#                 if child.dep_ == "cj" and child.pos_ == "NOUN":
#                     all_objects.append(child)
        
#         for obj in all_objects:
#             relations.append({
#                 "subjekt": subjekt.lemma_.lower(),
#                 "verb": token.lemma_.lower(),
#                 "objekt": obj.lemma_.lower()
#             })
    
#     return relations



In [85]:
# function declarations: homonym resolution

def build_noun_table(doc):
    seen = {}
    for sent in doc.sents:
        for token in sent:
            if token.pos_ == "NOUN":
                lemma = token.lemma_.lower()
                
                if lemma not in seen:
                    seen[lemma] = {
                        "id": len(seen),
                        "lemma": lemma,
                        "variants": set(),
                        "contexts": [],
                        "cluster_id": None
                    }
                
                seen[lemma]["variants"].add(token.text)
                seen[lemma]["contexts"].append({
                    "sent": sent.text.strip(),
                    "token_idx": token.i
                })
    
    return seen


def get_contextual_embedding(word, sentence, tokenizer, model, max_length=512):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=max_length)
    tokens = tokenizer.tokenize(sentence, truncation=True, max_length=max_length)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    word_tokens = tokenizer.tokenize(word)
    
    context_indices = [
        i + 1 for i, token in enumerate(tokens)
        if token not in word_tokens
    ]
    
    if not context_indices:
        return outputs.last_hidden_state[0][0].numpy()
    
    context_embeddings = outputs.last_hidden_state[0][context_indices]
    return context_embeddings.mean(dim=0).numpy()


def get_window_context(token_idx, sents, window=2):
    # sents wird jetzt übergeben, nicht neu berechnet
    sent_idx = next(
        (i for i, sent in enumerate(sents) if sent.start <= token_idx < sent.end),
        None
    )
    
    if sent_idx is None:
        return " ".join(s.text for s in sents)
    
    start = max(0, sent_idx - window)
    end = min(len(sents), sent_idx + window + 1)
    
    return " ".join(sent.text for sent in sents[start:end])


def split_homonymes(noun_table, doc, tokenizer, model, threshold=0.85):
    sents = list(doc.sents)  # einmal berechnen
    result = {}
    
    for lemma, entry in noun_table.items():
        contexts = entry["contexts"]
        
        if len(contexts) <= 1:
            result[lemma] = entry
            continue
        
        embeddings = np.array([
            get_contextual_embedding(
                lemma,
                get_window_context(ctx["token_idx"], sents),
                tokenizer,
                model
            )
            for ctx in contexts
        ])
        
        sim_matrix = cosine_similarity(embeddings)
        sims = sim_matrix[np.triu_indices(len(embeddings), k=1)]
        
        if len(sims) == 0:
            result[lemma] = entry
            continue
        
        threshold = sims.mean() - sims.std()
        
        groups = []
        for i, emb in enumerate(embeddings):
            placed = False
            for group in groups:
                sim = cosine_similarity([emb], [embeddings[group[0]]])[0][0]
                if sim >= threshold:
                    group.append(i)
                    placed = True
                    break
            if not placed:
                groups.append([i])
        
        if len(groups) == 1:
            result[lemma] = entry
        else:
            for idx, group in enumerate(groups):
                new_key = f"{lemma}_{idx}"
                result[new_key] = {
                    "lemma": lemma,
                    "variants": entry["variants"],
                    "contexts": [contexts[i] for i in group]
                }
    
    return result


def get_noun_reference(token, sent):
    # 1. Genitiv-Attribut
    genitiv = [c for c in token.children if c.dep_ == "ag"]
    if genitiv:
        return genitiv[0].lemma_.lower()
    
    # 2. "von + Nomen"
    for t in sent:
        if t.lemma_.lower() == "von":
            for child in t.children:
                if child.pos_ == "NOUN" and child.i < token.i:
                    return child.lemma_.lower()
    
    # 3. Subjekt am gleichen Verb
    verb = token.head
    if verb.dep_ == "cj":
        verb = verb.head
    if verb.dep_ == "cj":
        verb = verb.head
    
    if verb.pos_ in ("VERB", "AUX"):
        subjects = [
            t for t in sent
            if t.dep_ == "sb" and t.head == verb and t.pos_ == "NOUN" and t != token
        ]
        if subjects:
            return subjects[0].lemma_.lower()
    
    return None


def resolve_ambiguous_nouns(doc, noun_table):
    ambiguous = {
        lemma for lemma, entry in noun_table.items()
        if len(entry["contexts"]) > 1
    }
    
    result = dict(noun_table)
    replaced_contexts = {lemma: set() for lemma in ambiguous}

    for sent in doc.sents:
        for token in sent:
            if token.pos_ != "NOUN":
                continue
            
            lemma = token.lemma_.lower()
            if lemma not in ambiguous:
                continue
            
            bezug = get_noun_reference(token, sent)
            
            if not bezug or bezug == lemma:
                continue
            if bezug not in noun_table:
                continue
            
            new_key = f"{bezug}_{lemma}"
            
            if new_key not in result:
                result[new_key] = {
                    "id": len(result),
                    "lemma": new_key,
                    "variants": {token.text},
                    "contexts": [{"sent": sent.text.strip(), "token_idx": token.i}],
                    "cluster_id": None,
                    "resolved": True
                }
            else:
                result[new_key]["variants"].add(token.text)
                result[new_key]["contexts"].append({
                    "sent": sent.text.strip(),
                    "token_idx": token.i
                })
            
            replaced_contexts[lemma].add(token.i)
    
    for lemma in ambiguous:
        if lemma not in result:
            continue
        original_contexts = [
            c for c in result[lemma]["contexts"]
            if c["token_idx"] not in replaced_contexts[lemma]
        ]
        if original_contexts:
            result[lemma]["contexts"] = original_contexts
        else:
            del result[lemma]
    
    return result


def replace_nouns(doc, noun_table):
    replacements = {}
    
    for key, entry in noun_table.items():
        for context in entry["contexts"]:
            idx = context["token_idx"]
            original_form = doc[idx].text
            original_lemma = doc[idx].lemma_.lower()
            
            if "_" in key:
                prefix, base = key.rsplit("_", 1)
                flex_suffix = original_form[len(original_lemma):]
                new_label = f"{prefix.title()}_{base.title()}{flex_suffix}"
            else:
                flex_suffix = original_form[len(original_lemma):]
                new_label = key.title() + flex_suffix
            
            replacements[idx] = new_label
    
    tokens = [token.text for token in doc]
    for idx, label in replacements.items():
        tokens[idx] = label
    
    return " ".join(tokens)


def debug_embeddings(word, noun_table, doc, tokenizer, model):
    entry = noun_table.get(word)
    if not entry:
        print(f"'{word}' nicht in noun_table")
        return
    
    sents = list(doc.sents)
    print(f"\n=== '{word}' – {len(entry['contexts'])} Vorkommen ===")
    
    embeddings = []
    for i, context in enumerate(entry["contexts"]):
        window = get_window_context(context["token_idx"], sents)
        emb = get_contextual_embedding(word, window, tokenizer, model)
        embeddings.append(emb)
        print(f"\nVorkommen {i}:")
        print(f"  Fenster: {window[:100]}...")
    
    print("\n=== Ähnlichkeiten ===")
    for i in range(len(embeddings)):
        for j in range(i+1, len(embeddings)):
            sim = cosine_similarity([embeddings[i]], [embeddings[j]])[0][0]
            print(f"  Vorkommen {i} vs {j}: {sim:.4f}")

In [ ]:
# function declarations: synonym clustering

load_dotenv()
token = os.environ.get("HF_TOKEN")

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

HF_MODEL = "llama3.1:8b" # "meta-llama/Llama-3.1-8B-Instruct:cerebras"  # HuggingFace Modell für Synonym-Clusterung

# client = OpenAI(
#     base_url="https://router.huggingface.co/v1",
#     api_key=HF_API_KEY
# )

from itertools import combinations

def cluster_synonyms_llm(noun_table, original_text):
    lemmas = [k for k in noun_table.keys() if "_" not in k]
    
    prompt = f"""Du gruppierst Wörter mit gleicher Bedeutung aus dem Text.

Hat nur ein Wort eine Bedeutung, gruppierst du es allein.
Haben mehrere Wörter die gleiche Bedeutung, gruppierst du sie zusammen.

Prüfe für jede mögliche Gruppe: Kannst du im Text das eine Wort durch das andere ersetzen ohne die Bedeutung zu ändern?
Wenn ja, dann bildest du eine Gruppe.
Wenn nein, dann bildest du keine Gruppe.

Originaltext:
{original_text}

Substantive: {lemmas}

Antworte NUR als JSON, ohne Erklärungen, ohne Markdown-Backticks:
{{
  "groups": [
    {{"label": "Bauteil", "members": ["bauteil", "teil"]}}
  ]
}}"""

    completion = client.chat.completions.create(
        model="llama3.1:8b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2048,
        temperature=0.1
    )
    
    text = completion.choices[0].message.content.strip()
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    start = text.find("{")
    end = text.rfind("}") + 1
    groups = json.loads(text[start:end])
    
    # Nur Gruppen mit mehr als einem Member behalten wenn Begründung sinnvoll
    # Gruppen mit einem Member immer behalten
    return groups


def apply_synonym_clusters(noun_table, clusters):
    """Merged Synonym-Einträge in der noun_table"""
    result = dict(noun_table)
    
    for group in clusters["groups"]:
        label = group["label"].lower()
        members = group["members"]
        
        if len(members) <= 1:
            continue
        
        # Alle Kontexte und Varianten zusammenführen
        merged_contexts = []
        merged_variants = set()
        
        for member in members:
            if member in result:
                merged_contexts += result[member]["contexts"]
                merged_variants |= result[member]["variants"]
        
        # Neuen zusammengeführten Eintrag anlegen
        result[label] = {
            "id": result[members[0]]["id"] if members[0] in result else len(result),
            "lemma": label,
            "variants": merged_variants,
            "contexts": merged_contexts,
            "cluster_id": None
        }
        
        # Alte Einträge entfernen
        for member in members:
            if member in result and member != label:
                del result[member]
    
    return result

In [87]:
# function declarations: rule-based extraction 2


def extract_subjects(token, verb_groups):
    """Extrahiert Subjekte und löst Konjunktionen auf"""
    
    def collect_conjunctions(node, verb_idx):
        """Rekursiv alle konjunktiv verbundenen Nomen sammeln"""
        verb_groups[verb_idx]["subjects"].append(node)
        for child in node.children:
            if child.dep_ == "cj" and child.pos_ == "NOUN":
                collect_conjunctions(child, verb_idx)
    
    if token.pos_ == "PRON":
        head = token.head
        while head.dep_ not in ("rc", "ROOT"):
            head = head.head
        if head.dep_ == "rc":
            subj = head.head
            collect_conjunctions(subj, subj.head.i)
    else:
        collect_conjunctions(token, token.head.i)


def extract_objects(token, verb_groups):
    """Extrahiert Objekte und löst Konjunktionen auf"""
    verb_groups[token.head.i]["objects"].append(token)
    for child in token.children:
        if child.dep_ == "cj" and child.pos_ == "NOUN":
            verb_groups[token.head.i]["objects"].append(child)
        if child.dep_ == "cd" and child.pos_ == "CCONJ":
            for conj_child in child.children:
                if conj_child.pos_ == "NOUN" and conj_child.dep_ in ("nk", "da", "cj"):
                    verb_groups[token.head.i]["objects"].append(conj_child)


def build_verb_groups(sent):
    """Gruppiert Subjekte und Objekte nach ihrem Verb"""
    verb_groups = defaultdict(lambda: {"subjects": [], "objects": []})
    
    for token in sent:
        if token.dep_ in ("sb", "nsubj"): ###
            extract_subjects(token, verb_groups)
        elif token.dep_ in ("oa", "obj") or ((token.dep_ in ("nk", "da")) and token.pos_ == "NOUN"):
            extract_objects(token, verb_groups)
    
    return verb_groups


def build_relations(verb_groups, doc, entities):
    """Erstellt Relationen aus Verb-Gruppen"""
    relations = []
    
    for verb_idx, group in verb_groups.items():
        verb = doc[verb_idx].lemma_
        for subject in group["subjects"]:
            for obj in group["objects"]:
                subj_id = get_entity_id(subject, entities)
                obj_id = get_entity_id(obj, entities)
                
                subject_card = get_cardinality(subject)
                object_card = get_cardinality(obj)
                
                relations.append({
                    "subject": subj_id,
                    "predicate": verb,
                    "object": obj_id,
                    "cardinality": {
                        "subject": subject_card,
                        "object": object_card
                    }
                })
    
    return relations

In [88]:
# function declarations: komplexity reduction

def merge_relations(relations):
    """
    Checks whether two relations have the same predicate.
    If subject/object are the same or swapped:
    - Merge cardinalities using max() (n > 1)
    - Delete duplicate
    """
    def card_max(a, b):
        """n wins over 1"""
        return "n" if "n" in (a, b) else "1"

    to_delete = set()

    for i, r1 in enumerate(relations):
        if i in to_delete:
            continue

        for j, r2 in enumerate(relations):
            if j <= i or j in to_delete:
                continue

            # checks if same predicate
            if r1["predicate"] != r2["predicate"]:
                continue

            same   = r1["subject"] == r2["subject"] and r1["object"] == r2["object"]
            swapped = r1["subject"] == r2["object"]  and r1["object"] == r2["subject"]

            if same:
                # if same, merge cardinalities directly
                r1["cardinality"]["subject"] = card_max(
                    r1["cardinality"]["subject"],
                    r2["cardinality"]["subject"]
                )
                r1["cardinality"]["object"] = card_max(
                    r1["cardinality"]["object"],
                    r2["cardinality"]["object"]
                )
                to_delete.add(j)

            elif swapped:
                # if swapped, merge cardinalities crosswise
                r1["cardinality"]["subject"] = card_max(
                    r1["cardinality"]["subject"],
                    r2["cardinality"]["object"]     # crosswise merge
                )
                r1["cardinality"]["object"] = card_max(
                    r1["cardinality"]["object"],
                    r2["cardinality"]["subject"]    # crosswise merge
                )
                to_delete.add(j)

    return [r for i, r in enumerate(relations) if i not in to_delete]

# Entity to attribute
def collapse_weak_entities(entities, relations):
    """
    Removes weak entities (attribute candidates):
    - has exactly 1 relation
    - cardinality on its side is ‘1’
    - has no attributes of its own
        - is deleted, relation is deleted, its type is appended as an attribute to the other entity
    """
    to_delete_entities = set()
    to_delete_relations = set()

    # Index: entity_id → all relations + side
    from collections import defaultdict
    entity_relations = defaultdict(list)  # id → [(rel_index, side)]
    for i, rel in enumerate(relations):
        entity_relations[rel["subject"]].append((i, "subject"))
        entity_relations[rel["object"]].append((i, "object"))

    for key, entity in entities.items():
        eid = entity["id"]

        if len(entity_relations[eid]) != 1:
            continue

        rel_index, side = entity_relations[eid][0]
        rel = relations[rel_index]

        if rel["cardinality"][side] != "1":
            continue

        if entity.get("attributes"):
            continue

        other_side = "object" if side == "subject" else "subject"
        other_id = rel[other_side]

        other_key = next((k for k, e in entities.items() if e["id"] == other_id), None)
        if other_key is None:
            continue

        entities[other_key].setdefault("attributes", []).append(entity["type"])

        to_delete_entities.add(key)
        to_delete_relations.add(rel_index)

    for key in to_delete_entities:
        del entities[key]

    relations[:] = [r for i, r in enumerate(relations) if i not in to_delete_relations]

    return entities, relations

# remove duplicate annotions
# def remove_duplicate_annotations(relations):
    # unique_relations = []
    # seen = set()

    # for rel in relations:
    #     rel_tuple = (rel["subjekt"], rel["verb"], rel["objekt"])
    #     if rel_tuple not in seen:
    #         seen.add(rel_tuple)
    #         unique_relations.append(rel)

    # return unique_relations


In [89]:
# def replace_homonymes(doc, noun_table):
#     replacements = {}
    
#     for key, entry in noun_table.items():
#         # Fix 3: key as replacement label
#         for context in entry["contexts"]:
#             replacements[context["token_idx"]] = key
    
#     tokens = [token.text for token in doc]
#     for idx, label in replacements.items():
#         tokens[idx] = label
    
#     return " ".join(tokens)

In [90]:
# variable declaration
nlp = spacy.load("de_core_news_lg")

# Limitations: Need same nouns and same verb in same relation. Change Synonyms and Homonymes.
# Perfect text example:
text = """Ein Kunde kann mehrere Bestellungen aufgeben.
Eine Bestellung wird von genau einem Kunden aufgegeben.
Eine Bestellung enthält mehrere Produkte.
Ein Produkt ist in mehreren Bestellungen enthalten.
Zu einem Kunden gehört genau eine Lieferadresse.
Eine Lieferadresse gehört genau einem Kunden.
Ein Produkt kann eine Beschreibung haben."""

# only nessessery information:
text = """Ein Kunde kann mehrere Bestellungen aufgeben.
Eine Bestellung enthält mehrere Produkte, welche in mehreren Bestellungen enthalten sein können.
Zu einem Kunden gehört genau eine Lieferadresse.
Ein Produkt kann eine Beschreibung haben."""

# phrase "in der Regel" (Redewendung)
text = """Ein Fluss mündet maximal in ein Meer. In ein Meer mündet mindestens ein Fluss, in der Regel aber mehrere Flüsse."""

# real exercise
text = """Jedes Bauteil, das verwendet wird, hat eine eindeutige Nummer, ein Fertigungsdatum und eine
Bezeichnung, die allerdings für mehrere verschiedene Bauteile gleich sein kann.
Von jedem Teil werden außerdem der Name des Herstellers, der Einkaufspreis
pro Stück und der am Lager vorhandene Vorrat gespeichert. Jedes
herzustellende Gerät hat eine eindeutige Bezeichnung. Auch von jedem schon
gefertigten Gerätetyp soll der aktuelle Lagerbestand gespeichert werden, ebenso
wie der Verkaufspreis des Gerätes. In unserem fiktiven Betrieb gilt die Regelung,
dass Maschinen, die mehr als 1000,- EUR kosten, unentgeltlich an die Kunden
ausgeliefert werden; für Geräte, die weniger kosten, ist zusätzlich zum Preis eine
gerätespezifische Anliefergebühr zu entrichten. In der Datenbank ist ebenfalls zu
speichern, welche Bauteile für welche Geräte benötigt werden. Es gibt Bauteile,
die für mehrere Geräte verwendet werden. Von jedem Kunden werden der
Name, die Adresse und die Branche gespeichert. Es kann verschiedene Kunden
mit demselben Namen oder derselben Adresse geben. Außerdem ist zu jedem
Kunden vermerkt, wer aus unserer Firma für die entsprechende
Kundenbetreuung zuständig ist. Natürlich ist auch zu speichern, welche Kunden
mit welchen Geräten beliefert werden. Es kann sein, dass gewissen Kunden für
bestimmte Geräte Sonderkonditionen eingeräumt worden sind, dies soll ggf.
ebenfalls in der Datenbank vermerkt werden."""

# real exercise reduced to relevant sentences only
text = """Jedes Bauteil, das verwendet wird, hat eine eindeutige Nummer, ein Fertigungsdatum und eine
Bezeichnung, die allerdings für mehrere verschiedene Bauteile gleich sein kann.
Von jedem Teil werden außerdem der Name des Herstellers, der Einkaufspreis
pro Stück und der am Lager vorhandene Vorrat gespeichert. Jedes
herzustellende Gerät hat eine eindeutige Bezeichnung. Auch von jedem schon
gefertigten Gerätetyp soll der aktuelle Lagerbestand gespeichert werden, ebenso
wie der Verkaufspreis des Gerätes. In der Datenbank ist ebenfalls zu
speichern, welche Bauteile für welche Geräte benötigt werden. Es gibt Bauteile,
die für mehrere Geräte verwendet werden. Von jedem Kunden werden der
Name, die Adresse und die Branche gespeichert. Es kann verschiedene Kunden
mit demselben Namen oder derselben Adresse geben. Außerdem ist zu jedem
Kunden vermerkt, wer aus unserer Firma für die entsprechende
Kundenbetreuung zuständig ist. Natürlich ist auch zu speichern, welche Kunden
mit welchen Geräten beliefert werden. Es kann sein, dass gewissen Kunden für
bestimmte Geräte Sonderkonditionen eingeräumt worden sind, dies soll ggf.
ebenfalls in der Datenbank vermerkt werden."""

#text = " ".join(text.split())
doc = nlp(text)

entities = {}
relations = []

entity_id_counter = 1


In [91]:
# llm model
# contextuelles Embedding - gleiches Wort, verschiedener context = verschiedene Vektoren
model = "deepset/gbert-base"
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModel.from_pretrained(model)



In [92]:
# homoym resolution
# Phase 1: Rule-based
noun_table = build_noun_table(doc)
noun_table = resolve_ambiguous_nouns(doc, noun_table)
text_phase1 = replace_nouns(doc, noun_table)

print("=== Nach Rule-based ===")
print(text_phase1)
with open("output.txt", "w", encoding="utf-8") as f:
    f.write(text_phase1)

# Phase 2: Embedding-based
# doc2 = nlp(text_phase1)
# noun_table2 = build_noun_table(doc2)
# noun_table2 = split_homonymes(noun_table2, doc2, tokenizer, model)
# text_phase2 = replace_nouns(doc2, noun_table2)

# print("\n=== Nach Embedding-based ===")
# print(text_phase2)

=== Nach Rule-based ===
Jedes Bauteil , das verwendet wird , hat eine eindeutige Nummer , ein Fertigungsdatum und eine 
 Bezeichnung , die allerdings für mehrere verschiedene Bauteile gleich sein kann . 
 Von jedem Teil werden außerdem der Hersteller_Name des Herstellers , der Einkaufspreis 
 pro Stück und der am Lager vorhandene Vorrat gespeichert . Jedes 
 herzustellende Gerät hat eine eindeutige Gerät_Bezeichnung . Auch von jedem schon 
 gefertigten Gerätetyp soll der aktuelle Lagerbestand gespeichert werden , ebenso 
 wie der Verkaufspreis des Gerätetyp_Gerätes . In der Datenbank ist ebenfalls zu 
 speichern , welche Bauteile für welche Geräte benötigt werden . Es gibt Bauteile , 
 die für mehrere Geräte verwendet werden . Von jedem Kunden werden der 
 Kunde_Name , die Kunde_Adresse und die Branche gespeichert . Es kann verschiedene Kunden 
 mit demselben Namen oder derselben Adresse geben . Außerdem ist zu jedem 
 Kunden vermerkt , wer aus unserer Firma für die entsprechende 
 Kun

In [93]:
# synonym clustering

# Nach build_noun_table und resolve_ambiguous_nouns:
noun_table = build_noun_table(doc)
noun_table = resolve_ambiguous_nouns(doc, noun_table)

# Synonym-Clustering
clusters = cluster_synonyms_llm(noun_table, text)
print(json.dumps(clusters, indent=2, ensure_ascii=False))
noun_table = apply_synonym_clusters(noun_table, clusters)
text_phase2 = replace_nouns(doc, noun_table)
print(text_phase2)

{
  "groups": [
    {
      "label": "Bauteil",
      "members": [
        "bauteil",
        "teil"
      ]
    },
    {
      "label": "Fertigungsdatum",
      "members": [
        "fertigungsdatum"
      ]
    },
    {
      "label": "Bezeichnung",
      "members": [
        "bezeichnung"
      ]
    },
    {
      "label": "Name",
      "members": [
        "name"
      ]
    },
    {
      "label": "Hersteller",
      "members": [
        "hersteller"
      ]
    },
    {
      "label": "Preis",
      "members": [
        "einkaufspreis",
        "verkaufspreis"
      ]
    },
    {
      "label": "Lager",
      "members": [
        "lager",
        "vorrat",
        "lagerbestand"
      ]
    },
    {
      "label": "Gerät",
      "members": [
        "gerät",
        "gerätetyp"
      ]
    },
    {
      "label": "Datenbank",
      "members": [
        "datenbank"
      ]
    },
    {
      "label": "Kunde",
      "members": [
        "kunde"
      ]
    },
    {
      "label":

In [94]:
# Main - extraction loop
doc = nlp(text_phase2) #_phase1)
# Extract entities and relations
for sent in doc.sents:  # Loop through sentences
    
    if not has_root_verb(sent):
        continue
    if not is_relevant(sent):
        continue

    # Extract subject, verb, and object
    verb = None # get_main_verb(sent)
    verb_lemma = None
    subject_card = None
    object_card = None
    attribute = None
    verb_groups = defaultdict(lambda: {"subjects": [], "objects": []})

    # Identify subject and object based on dependency labels
    
    for token in sent:
        if token.dep_ in ("sb", "nsubj"):
            verb_idx = get_verb_root(token)
            get_subjects(token, verb_groups, verb_idx)
        elif token.dep_ in ("oa", "obj") or ((token.dep_ in ("nk", "da")) and token.pos_ == "NOUN"):
            verb_idx = get_verb_root(token)
            get_objects(token, verb_groups, verb_idx)

    # Only create a relation if we have a valid subject, verb, and object
    for verb_idx, group in verb_groups.items():
        verb_lemma = doc[verb_idx].lemma_
        if len(group["subjects"]) > 0 and len(group["objects"]) > 0:
            for subject in group["subjects"]:
                for obj in group["objects"]:
                    subj_id = get_entity_id(subject, entities)
                    obj_id = get_entity_id(obj, entities)
                    subject_card = get_cardinality(subject)
                    object_card = get_cardinality(obj)
                    relations.append({
                        "subject": subj_id,
                        "predicate": verb_lemma,
                        "object": obj_id,
                        "cardinality": {
                            "subject": subject_card,
                            "object": object_card
                        }
                    })

# Merge duplicate relations
relations = merge_relations(relations)

# Collapse weak entities into attributes
entities, relations = collapse_weak_entities(entities, relations)

output = {
    "entities": list(entities.values()),
    "relations": relations
}

# print(json.dumps(output, indent=2, ensure_ascii=False))
with open('data.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)


In [95]:
# # Main - extraction loop
# doc = nlp(text_phase1)
# # Extract entities and relations
# for sent in doc.sents:
#     if not has_root_verb(sent):
#         continue
#     # if not is_relevant(sent):
#     #     continue
    
#     verb_groups = build_verb_groups(sent)
#     relations += build_relations(verb_groups, doc, entities)
    
# # Merge duplicate relations
# # relations = merge_relations(relations)

# # Collapse weak entities into attributes
# # entities, relations = collapse_weak_entities(entities, relations)

# output = {
#     "entities": list(entities.values()),
#     "relations": relations
# }

# # print(json.dumps(output, indent=2, ensure_ascii=False))
# with open('data.json', 'w', encoding='utf-8') as f:
#     json.dump(output, f, indent=2, ensure_ascii=False)


In [96]:
for token in doc:
    print(token.text, token.dep_, token.pos_)

Jedes nk DET
Bauteil sb NOUN
, punct PUNCT
das sb PRON
verwendet oc VERB
wird rc AUX
, punct PUNCT
hat ROOT VERB
eine nk DET
eindeutige nk ADJ
Nummer oa NOUN
, punct PUNCT
ein nk DET
Fertigungsdatum cj NOUN
und cd CCONJ
eine nk DET

  dep SPACE
Bezeichnung cj NOUN
, punct PUNCT
die sb PRON
allerdings mo ADV
für mo ADP
mehrere nk DET
verschiedene nk ADJ
Bauteile nk NOUN
gleich mo ADV
sein oc AUX
kann rc AUX
. punct PUNCT

  dep SPACE
Von mo ADP
jedem nk DET
Bauteil nk NOUN
werden ROOT AUX
außerdem mo ADV
der nk DET
Hersteller_Name sb NOUN
des nk DET
Herstellers ag NOUN
, punct PUNCT
der nk DET
Preis sb NOUN

  dep SPACE
pro mnr ADP
Stück nk NOUN
und cd CCONJ
der nk DET
am mo ADP
Lager nk NOUN
vorhandene nk ADJ
Lager cj NOUN
gespeichert oc VERB
. punct PUNCT
Jedes nk DET

  dep SPACE
herzustellende nk ADJ
Gerät sb NOUN
hat ROOT VERB
eine nk DET
eindeutige nk ADJ
Gerät_Bezeichnung oa NOUN
. punct PUNCT
Auch mo ADV
von mo ADP
jedem nk DET
schon mo ADV

  dep SPACE
gefertigten nk ADJ
Gerät 

In [97]:
import spacy
from spacy import displacy

text = nlp("Jedes Bauteil, das verwendet wird, hat eine eindeutige Nummer und eine Bezeichnung, die allerdings für mehrere verschiedene Bauteile gleich sein kann.")

#displacy.serve(text, style="dep")
svg = displacy.render(doc, style="dep", jupyter=False)

with open("dependency.svg", "w", encoding="utf-8") as f:
    f.write(svg)

# tmp

In [98]:
tmp_doc = nlp("Eine Bestellung enthält mehrere Produkte.")
print("tmp_doc:", type(tmp_doc))
for token in tmp_doc:
    print(token.lemma_)
#    print(token.text, token.dep_, token.pos_)

tmp_doc: <class 'spacy.tokens.doc.Doc'>
ein
Bestellung
enthalten
mehrere
Produkt
--


In [99]:
def extract_kern(text):
    doc = nlp(text)
    result = {"subjekt": None, "verb": None, "objekt": None}

    hilfsverben = {"sein", "werden", "haben"}

    for token in doc:
        # Verb: ROOT, aber bei Hilfsverb → pd bevorzugen
        if token.dep_ == "ROOT":
            if token.lemma_ in hilfsverben:
                # Suche nach Prädikativ
                pd = next((c for c in token.children if c.dep_ == "pd"), None)
                result["verb"] = pd.lemma_ if pd else token.lemma_
            else:
                result["verb"] = token.lemma_

        # Subjekt
        if token.dep_ in ("sb", "nsubj") and token.pos_ in ("NOUN", "PROPN"):
            result["subjekt"] = token.lemma_

        # Direktes Objekt
        if token.dep_ in ("oa", "oc", "dobj") and token.pos_ in ("NOUN", "PROPN"):
            result["objekt"] = token.lemma_

        # Objekt in Präpositionalphrase (z.B. "in Bestellungen")
        if token.dep_ in ("mo", "pg") and token.pos_ == "ADP":
            for child in token.children:
                if child.pos_ in ("NOUN", "PROPN"):
                    result["objekt"] = child.lemma_

    return result

sätze = [
    "Ein Produkt ist in mehreren Bestellungen enthalten."
]

for sent in sätze:
    r = extract_kern(sent)
    print(f"{r['subjekt']} {r['verb']} {r['objekt']}")


Produkt enthalten Bestellung


In [100]:
sätze = [
    "Ein Kunde kann mehrere Bestellungen aufgeben.",
    "Eine Bestellung gehört genau einem Kunden.",
    "Eine Bestellung enthält mehrere Produkte.",
    "Ein Produkt ist in mehreren Bestellungen enthalten.",
]

for sent in sätze:
    doc = nlp(sent)
    for token in doc:
        if token.pos_ in ("NOUN", "PROPN"):
            card = get_cardinality(token)
            print(f"{token.text:<15} dep={token.dep_:<6} card={card}")
    print()

Kunde           dep=sb     card=1
Bestellungen    dep=oa     card=n

Bestellung      dep=sb     card=1
Kunden          dep=da     card=1

Bestellung      dep=sb     card=1
Produkte        dep=oa     card=n

Produkt         dep=sb     card=1
Bestellungen    dep=nk     card=n

